# Imports & Functions

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from sklearn.ensemble import HistGradientBoostingRegressor
import xgboost as xgb


In [ ]:
def remove_top_1_percent_outliers(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    df_clean = df.copy()
    for col in features:
        lower_bound = np.percentile(df_clean[col], 1)
        upper_bound = np.percentile(df_clean[col], 99)
        
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
    
    return df_clean

def remove_outliers(df, target, threshold=0.012):
    """Removes the extreme outliers"""
    lower_bound = target.quantile(threshold)
    upper_bound = target.quantile(1 - threshold)
    
    mask = (target >= lower_bound) & (target <= upper_bound)
    return df[mask], target[mask]

def frequency_encoding(df_to_modify: pd.DataFrame, df_initial: pd.DataFrame, column: str) -> pd.DataFrame:
    freq_map = df_initial[column].value_counts()
    df_to_modify[column + "_freq"] = df_initial[column].map(freq_map).fillna(0)
    return df_to_modify

# Read files

In [254]:
x_test_file = pd.read_csv(r"/Users/raphaelcorchia/Documents/Pro/Dauphine/Master 203/M2/S2/Machine Learning/Challenge/x_test_final.csv")
x_train_file = pd.read_csv(r"/Users/raphaelcorchia/Documents/Pro/Dauphine/Master 203/M2/S2/Machine Learning/Challenge/x_train_final.csv")
y_sample = pd.read_csv(r"/Users/raphaelcorchia/Documents/Pro/Dauphine/Master 203/M2/S2/Machine Learning/Challenge/y_sample_final.csv")
y_train_file = pd.read_csv(r"/Users/raphaelcorchia/Documents/Pro/Dauphine/Master 203/M2/S2/Machine Learning/Challenge/y_train_final_j5KGWWK.csv")

# Data & Engineering

### Train Data

In [255]:
###### Train through Sklearn #####

x_train, x_val, y_train, y_val = train_test_split(x_train_file, y_train_file, test_size=0.1, random_state=54)
y_train = y_train.iloc[:, -1]
y_val = y_val.iloc[:, -1]

In [234]:
##### Train through simple way #####

split_idx = int(len(x_train_file) * 0.9)
x_train, x_val = x_train_file.iloc[:split_idx], x_train_file.iloc[split_idx:]
y_train, y_val = y_train_file.iloc[:split_idx], y_train_file.iloc[split_idx:]

y_train = y_train.iloc[:, -1]
y_val = y_val.iloc[:, -1]

### Outliers & Encoding

In [ ]:
###################################
##### OLD VERSION OF OUTLIERS #####
###################################

outlier_features = ["p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]
x_train = remove_top_1_percent_outliers(x_train, outlier_features)
y_train = y_train.loc[x_train.index]

In [256]:
##### Encoding #####

features_to_keep = ["p2q0", "p3q0", "p4q0", "p0q2", "p0q3", "p0q4"]
X_train = x_train[features_to_keep].copy()
X_val = x_val[features_to_keep].copy()
X_test = x_test_file[features_to_keep].copy()
Y_train = y_train.copy()

# Feature Engineering: Encode "gare" and "arret" column
gare_counts = x_train['gare'].value_counts()
X_train['gare_encoded'] = x_train['gare'].map(gare_counts)
X_val['gare_encoded'] = x_val['gare'].map(gare_counts).fillna(0)
X_test['gare_encoded'] = x_test_file['gare'].map(gare_counts).fillna(0)

arret_counts = x_train['arret'].value_counts()
X_train['arret_encoded'] = x_train['arret'].map(arret_counts)
X_val['arret_encoded'] = x_val['arret'].map(arret_counts).fillna(0)
X_test['arret_encoded'] = x_test_file['arret'].map(arret_counts).fillna(0)

In [257]:
##### Remove Outliers #####

X_train_clean, Y_train_clean = remove_outliers(X_train, Y_train, threshold=0.012)

# Ensure validation set has same columns
X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Weighted Average Model

In [ ]:
## First method with weighted mean

x_test_final_test = x_test_file.copy()
x_test_final_test["p0q0"] = (0.6 * x_test_final_test["p0q2"] + 0.3 * x_test_final_test["p0q3"] + 0.1 * x_test_final_test["p0q4"]).round(0)
y_first_test = x_test_final_test["p0q0"]
y_first_test.to_csv("Weighted_Average.csv")

# Random Forest Model

In [ ]:
# Initialise the model
rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Train the model
rf.fit(X_train, Y_train.values.ravel())

# Predecit on all values
y_pred = rf.predict(X_val)

# Evaluate the model
mae = mean_absolute_error(y_val, y_pred)
print(f"Erreur absolue moyenne (MAE) : {mae}")


Erreur absolue moyenne (MAE) : 0.7775141810225323


In [ ]:
y_test_pred = rf.predict(X_test)
submission = pd.DataFrame({'p0q0': y_test_pred})
submission.to_csv("submission_RF.csv", index=False)

# HGB Model

In [ ]:
# Boosted gradient model
hgb = HistGradientBoostingRegressor(
    max_iter=1000,
    max_depth=50,
    learning_rate=0.2,
    max_bins=255,
    l2_regularization=1,
    random_state=42)

hgb.fit(X_train, Y_train)

y_pred = hgb.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
print(f"Erreur absolue moyenne (MAE) : {mae}")

/Users/raphaelcorchia/Documents/Pro/Dauphine/Master 203/M2/S2/Machine Learning/Challenge/sncf/lib/python3.11/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Erreur absolue moyenne (MAE) : 0.7775059639032569


In [ ]:
# Predict the test file
y_test_pred = hgb.predict(X_test)

# Create the good file
submission = pd.DataFrame({
    "Unnamed: 0": y_sample["Unnamed: 0"],
    "p0q0": y_test_pred
})

submission.to_csv("submission_hgb.csv", index=False)

# XGBoost Model

In [ ]:
# Initialize the model
xgb_model = xgb.XGBRegressor(
    n_estimators=1400,   # Number of trees (can be tuned)
    max_depth=12,        # Depth of each tree
    learning_rate=0.009,  # Step size (can be tuned)
    subsample=0.9,      # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    random_state=54,
    n_jobs=-1           # Use all CPU cores
)

# Fit the model
xgb_model.fit(X_train_clean, Y_train_clean)

# Predict
y_pred = xgb_model.predict(X_val)
y_pred = y_pred.round(0).astype(int)
y_pred = pd.DataFrame(y_pred)

# Evaluate
mae = mean_absolute_error(y_val, y_pred)
print(f"Erreur absolue moyenne (MAE) avec XGBoost : {mae}")


Erreur absolue moyenne (MAE) avec XGBoost : 0.6360843436689796


In [ ]:
# Predict on test
y_test_pred = xgb_model.predict(X_test)
y_test_pred = y_test_pred.round(0).astype(int)
y_test_pred = pd.DataFrame(y_test_pred)

# Save in a csv file
y_test_pred.to_csv("submission_xgboost8.csv")

print("Fichier de soumission sauvegardé : submission_xgboost8.csv")


Fichier de soumission sauvegardé : submission_xgboost8.csv
